In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Passes de transpilador com IA
Os passes de transpilador com IA são passes que funcionam como substituição direta dos passes "tradicionais" do Qiskit para algumas tarefas de transpilação. Com frequência, eles produzem resultados melhores do que os algoritmos heurísticos existentes (como menor profundidade e contagem de CNOTs), mas também são muito mais rápidos do que algoritmos de otimização como os resolvedores de satisfatibilidade booleana. Os passes de transpilador com IA podem ser executados no seu ambiente local ou na nuvem por meio do Qiskit Transpiler Service, caso você faça parte do IBM Quantum&reg; Premium Plan, Flex Plan ou On-Prem (via IBM Quantum Platform API) Plan.

> **Note:** Os passes de transpilador com IA estão em status de lançamento beta, sujeitos a alterações.
>     Se você tiver comentários ou quiser entrar em contato com a equipe de desenvolvimento, use este [canal do Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Os seguintes passes estão disponíveis atualmente:

**Passes de roteamento**
 - `AIRouting`: Seleção de layout e roteamento de circuito

**Passes de síntese de circuito**
 - `AICliffordSynthesis`: Síntese de circuito Clifford
 - `AILinearFunctionSynthesis`: Síntese de circuito de função linear
 - `AIPermutationSynthesis`: Síntese de circuito de permutação
 - `AIPauliNetworkSynthesis`: Síntese de circuito de rede de Pauli

Para usar os passes de transpilador com IA, primeiro [instale o pacote `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Visite a [documentação da API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) para obter mais informações sobre as diferentes opções disponíveis.

## Executar os passes de transpilador com IA localmente ou na nuvem
Se você quiser usar os passes de transpilador com IA no seu ambiente local gratuitamente, instale o `qiskit-ibm-transpiler` com algumas dependências extras da seguinte forma:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Sem essas dependências extras, os passes de transpilador com IA são executados na nuvem por meio do Qiskit Transpiler Service (disponível apenas para usuários do IBM Quantum Premium Plan, Flex Plan ou On-Prem (via IBM Quantum Platform API) Plan). Após instalar as dependências extras, o modo padrão para executar os passes de transpilador com IA é usar sua máquina local.

## Passe de roteamento com IA
O passe `AIRouting` atua tanto como estágio de layout quanto como estágio de roteamento. Ele pode ser usado dentro de um `PassManager` da seguinte forma:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Aqui, o `backend` determina para qual mapa de acoplamento rotear, o `optimization_level` (1, 2 ou 3) determina o esforço computacional a ser empregado no processo (valores maiores geralmente produzem melhores resultados, mas levam mais tempo), e o `layout_mode` especifica como tratar a seleção de layout.
O `layout_mode` inclui as seguintes opções:

- `keep`: Respeita o layout definido pelos passes de transpilador anteriores (ou usa o layout trivial se não estiver definido). Tipicamente, é usado apenas quando o circuito deve ser executado em qubits específicos do dispositivo. Com frequência, produz resultados piores porque tem menos espaço para otimização.
- `improve`: Usa o layout definido pelos passes de transpilador anteriores como ponto de partida. É útil quando você tem uma boa estimativa inicial para o layout; por exemplo, para circuitos construídos de forma que sigam aproximadamente o mapa de acoplamento do dispositivo. Também é útil se você quiser experimentar outros passes de layout específicos combinados com o passe `AIRouting`.
- `optimize`: Este é o modo padrão. Funciona melhor para circuitos gerais nos quais você pode não ter boas estimativas de layout. Este modo ignora seleções de layout anteriores.
- `local_mode`: Este sinalizador determina onde o passe `AIRouting` é executado. Se `False`, o `AIRouting` é executado remotamente por meio do Qiskit Transpiler Service. Se `True`, o pacote tenta executar o passe no seu ambiente local, com fallback para o modo em nuvem caso as dependências necessárias não sejam encontradas.

## Passes de síntese de circuito com IA
Os passes de síntese de circuito com IA permitem otimizar partes de diferentes tipos de circuito ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Função Linear](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutação](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Rede de Pauli) por meio de re-síntese. Uma forma típica de usar o passe de síntese é a seguinte:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

A síntese respeita o mapa de acoplamento do dispositivo: ela pode ser executada com segurança após outros passes de roteamento sem perturbar o circuito, de modo que o circuito geral ainda seguirá as restrições do dispositivo. Por padrão, a síntese substituirá o sub-circuito original apenas se o sub-circuito sintetizado melhorar o original (atualmente verificando apenas a contagem de CNOTs), mas é possível forçar a substituição sempre definindo `replace_only_if_better=False`.

Os seguintes passes de síntese estão disponíveis em `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Síntese para circuitos [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (blocos de portas `H`, `S` e `CX`). Atualmente suporta blocos de até nove qubits.
- *AILinearFunctionSynthesis*: Síntese para circuitos de [Função Linear](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (blocos de portas `CX` e `SWAP`). Atualmente suporta blocos de até nove qubits.
- *AIPermutationSynthesis*: Síntese para circuitos de [Permutação](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (blocos de portas `SWAP`). Atualmente disponível para blocos de 65, 33 e 27 qubits.
- *AIPauliNetworkSynthesis*: Síntese para circuitos de Rede de Pauli (blocos de portas `H`, `S`, `SX`, `CX`, `RX`, `RY` e `RZ`). Atualmente suporta blocos de até seis qubits.

Esperamos aumentar gradualmente o tamanho dos blocos suportados.

Todos os passes usam um pool de threads para enviar várias solicitações em paralelo. Por padrão, o número máximo de threads é o número de núcleos mais quatro (valores padrão do objeto Python `ThreadPoolExecutor`). No entanto, você pode definir seu próprio valor com o argumento `max_threads` na instanciação do passe. Por exemplo, a linha a seguir instancia o passe `AILinearFunctionSynthesis`, permitindo que ele use no máximo 20 threads.